In [0]:
# Databricks Notebook
# Step 3: Three-table JOIN + Market Share + K-Means Clustering → Output final analysis dataset

from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

# ====================== 1. Read three processed datasets ======================
yellow = spark.read.parquet("/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/yellow_tripdata_with_time_features")
green = spark.read.parquet("/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/green_tripdata_with_time_features")
fhvhv = spark.read.parquet("/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/fhvhv_tripdata_with_time_features")

# ====================== 2. Count orders by zone + hour ======================
y = yellow.groupBy("PULocationID", "Hour").count().withColumnRenamed("count", "yellow_count")
g = green.groupBy("PULocationID", "Hour").count().withColumnRenamed("count", "green_count")
f = fhvhv.groupBy("PULocationID", "Hour").count().withColumnRenamed("count", "fhvhv_count")

# ====================== 3. FULL OUTER JOIN of three tables (core step) ======================
combined = y.join(g, ["PULocationID", "Hour"], "outer").join(f, ["PULocationID", "Hour"], "outer").fillna(0)

# ====================== ✅ Output Part 1: Three-table JOIN result dataset ======================
join_output = "/Volumes/workspace/6009小组作业/nyctlcdataset/step3_join_combined_result"
combined.write.mode("overwrite").parquet(join_output)
print("✅ Exported: Three-table JOIN result dataset")
print(f"Path: {join_output}")

# ====================== ✅ Check row count of three-table JOIN output ======================
# Read the JOIN result just output
df_join_check = spark.read.parquet("/Volumes/workspace/6009小组作业/nyctlcdataset/step3_join_combined_result")

# Print row count
print("\n======================================")
print("📊 Total rows after three-table JOIN: ", df_join_check.count())
print("======================================")

✅ 已输出：三表JOIN后数据集
路径：/Volumes/workspace/6009小组作业/nyctlcdataset/step3_join_combined_result

📊 三表 JOIN 后的数据集总行数： 6283


In [0]:
# ====================== 4. Calculate market share ======================
combined = combined.withColumn("total_count", F.col("yellow_count") + F.col("green_count") + F.col("fhvhv_count"))
combined = combined.withColumn("hvfhv_share", F.when(F.col("total_count")>0, F.col("fhvhv_count")/F.col("total_count")).otherwise(0))

# ====================== 5. Aggregate by Zone (one final row per Zone)======================
zone_df = combined.groupBy("PULocationID") \
    .agg(
        F.sum("yellow_count").alias("yellow_total"),
        F.sum("green_count").alias("green_total"),
        F.sum("fhvhv_count").alias("fhvhv_total"),
        F.sum("total_count").alias("total_orders"),
        F.avg("hvfhv_share").alias("avg_hvfhv_share")
    ).filter(F.col("total_orders") > 0)

# ====================== ✅ Output Part 2: Zone aggregation dataset before clustering ======================
zone_output = "/Volumes/workspace/6009小组作业/nyctlcdataset/step3_zone_aggregation_before_clustering"
zone_df.write.mode("overwrite").parquet(zone_output)
print("\n✅ Exported: Zone aggregation dataset before clustering")
print(f"Path: {zone_output}")

# ====================== ✅ Check: row count + column count ======================
# Read the file just output
df_check = spark.read.parquet(zone_output)

# Count rows
row_count = df_check.count()

# Count columns
col_count = len(df_check.columns)

# Print results
print("\n======================================")
print(f"📊 Zone aggregation dataset before clustering row count: {row_count}")
print(f"📊 Zone aggregation dataset before clustering column count: {col_count}")
print("======================================")


✅ 已输出：聚类前Zone聚合数据集
路径：/Volumes/workspace/6009小组作业/nyctlcdataset/step3_zone_aggregation_before_clustering

📊 聚类前Zone聚合数据集 行数：263
📊 聚类前Zone聚合数据集 列数：6


In [0]:
# ====================== 6. K-Means Clustering ======================
assembler = VectorAssembler(inputCols=["avg_hvfhv_share"], outputCol="features")
zone_features = assembler.transform(zone_df)

kmeans = KMeans(k=3, seed=42)
model = kmeans.fit(zone_features)
result = model.transform(zone_features)

# ====================== 7. ✅ Correct label mapping (fix embedded)======================
# Classify directly based on rideshare share, absolutely correct
result = result.withColumn("zone_type",
    F.when(F.col("avg_hvfhv_share") < 0.2, "Taxi Stronghold")       # Rideshare < 20%
    .when(F.col("avg_hvfhv_share") < 0.8, "Fierce Competition Zone")    # 20% ~ 80%
    .otherwise("Rideshare Dominated Zone")                                 # > 80%
)

# ====================== 8. Final dataset (keep only useful fields) ======================
final_dataset = result.select(
    "PULocationID",
    "yellow_total",
    "green_total",
    "fhvhv_total",
    "total_orders",
    "avg_hvfhv_share",
    "zone_type"
).orderBy("PULocationID")

# ====================== 9. Output final correct dataset ======================
output_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/final_zone_market_dataset_CORRECTED"
final_dataset.write.mode("overwrite").parquet(output_path)

print("🎉 Successfully exported the [100% correct] final analysis dataset!")
print(f"Path: {output_path}")
print(f"Number of zones: {final_dataset.count()}")
final_dataset.show(60, False)

🎉 成功输出【100%正确】最终分析数据集！
路径：/Volumes/workspace/6009小组作业/nyctlcdataset/final_zone_market_dataset_CORRECTED
区域数量：263
+------------+------------+-----------+-----------+------------+--------------------+--------------+
|PULocationID|yellow_total|green_total|fhvhv_total|total_orders|avg_hvfhv_share     |zone_type     |
+------------+------------+-----------+-----------+------------+--------------------+--------------+
|1           |1065        |1          |4          |1070        |0.007156421127009362|出租车坚守区  |
|2           |44          |0          |86         |130         |0.6505032845941937  |两者激烈竞争区|
|3           |1482        |6          |138657     |140145      |0.9897423421012673  |网约车统治区  |
|4           |37234       |0          |181087     |218321      |0.8467669618638682  |网约车统治区  |
|5           |3           |0          |17216      |17219       |0.9998698647629597  |网约车统治区  |
|6           |437         |0          |34745      |35182       |0.9891367421215272  |网约车统治区  |
|7           |1

In [0]:
final_dataset.groupBy("zone_type") \
             .agg(F.min("avg_hvfhv_share"), F.max("avg_hvfhv_share"), F.avg("avg_hvfhv_share")) \
             .show()

+--------------+--------------------+--------------------+--------------------+
|     zone_type|min(avg_hvfhv_share)|max(avg_hvfhv_share)|avg(avg_hvfhv_share)|
+--------------+--------------------+--------------------+--------------------+
|  出租车坚守区|1.431226908559865E-4|0.007156421127009362|0.003649771908932...|
|两者激烈竞争区|  0.2215826035418309|  0.7985550151166724|  0.5328841228297767|
|  网约车统治区|  0.8111880797828084|                 1.0|  0.9744937706944248|
+--------------+--------------------+--------------------+--------------------+

